# Clase 117 — Stochastic Depth, DropPath, LayerDrop

*Huang et al. 2016*: durante entrenamiento, droppea capas residuales enteras con prob `p_l`. En inference, escalá el output.

Implementamos sin torch — todo numpy + sklearn.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)

def stochastic_depth(x, p, training=True):
    """Si training: con prob p dropea (devuelve 0). Si no, escala por (1-p).
    Funciona para batch o single sample. p=0 → identidad.
    """
    if not training:
        return x * (1 - p)
    if np.random.rand() < p:
        return np.zeros_like(x)
    return x

## 1. ResNet block simulado + StochDepth

In [ ]:
def resnet_block_mock(x, layer_idx, p_drop=0.0, training=True):
    """3 conv layers mock (mat mul rand) + skip connection + stochastic depth."""
    rng = np.random.default_rng(layer_idx)
    W1 = rng.normal(0, 0.1, (x.shape[-1], x.shape[-1]))
    W2 = rng.normal(0, 0.1, (x.shape[-1], x.shape[-1]))
    W3 = rng.normal(0, 0.1, (x.shape[-1], x.shape[-1]))
    h = np.maximum(0, x @ W1)
    h = np.maximum(0, h @ W2)
    h = h @ W3
    h = stochastic_depth(h, p_drop, training=training)
    return x + h  # skip conn

x = np.random.randn(4, 16)
out = resnet_block_mock(x, 0, p_drop=0.3, training=True)
print('out shape:', out.shape)

## 2. Linear schedule: p_l = (l/L) * p_max

In [ ]:
L = 24; p_max = 0.2
p_l = np.array([(l/L)*p_max for l in range(1, L+1)])
plt.figure(figsize=(7, 3))
plt.plot(p_l, marker='o'); plt.xlabel('layer index l'); plt.ylabel('p_drop')
plt.title(f'Linear stochastic depth (L={L}, p_max={p_max})'); plt.grid(True)
plt.show()
print(f'capas tempranas: p≈{p_l[0]:.3f}; capas tardías: p≈{p_l[-1]:.3f}')
print(f'expected layers ejecutadas por forward pass: {(1-p_l).sum():.1f}/{L}')

## 3. Comparativa con/sin StochDepth en MLP (digits 8x8)

In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

X, y = load_digits(return_X_y=True)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42)
sc = StandardScaler().fit(Xtr); Xtr = sc.transform(Xtr); Xte = sc.transform(Xte)

# Baseline: MLP normal
m_base = MLPClassifier(hidden_layer_sizes=(64,64,64), max_iter=200, random_state=42).fit(Xtr, ytr)
acc_base = m_base.score(Xte, yte)

# Stoch depth simulado: agregamos ruido por feature dropping en input + early stop más agresivo
# (simulamos efecto regularizador)
rng2 = np.random.default_rng(42)
mask = (rng2.random(Xtr.shape) > 0.1).astype(float)
m_sd = MLPClassifier(hidden_layer_sizes=(64,64,64), max_iter=200, random_state=42).fit(Xtr*mask, ytr)
acc_sd = m_sd.score(Xte, yte)

print(f'baseline acc: {acc_base:.4f}')
print(f'con stoch-depth-like noise: {acc_sd:.4f}')

## 4. Forward pass con stochastic depth schedule

In [ ]:
x0 = np.random.randn(2, 16)
for trial in range(3):
    h = x0.copy(); activos = 0
    for l, p in enumerate(p_l):
        h_new = resnet_block_mock(h, l, p_drop=p, training=True)
        if not np.allclose(h_new, h):
            activos += 1
        h = h_new
    print(f'trial {trial}: capas activas = {activos}/{L}')

## 5. LayerDrop (Fan et al. 2019) y DropPath

- **LayerDrop**: variante para transformers — droppear capas enteras en entrenamiento permite *pruning* en inference (cortás layers sin re-entrenar).
- **DropPath**: el mismo concepto en Vision Transformers (`timm.layers.DropPath`).

```python
# timm.layers.DropPath equivalente
class DropPath(nn.Module):
    def __init__(self, p=0.):
        super().__init__(); self.p = p
    def forward(self, x):
        if self.p == 0. or not self.training: return x
        keep = 1 - self.p
        mask = torch.bernoulli(torch.full((x.size(0),) + (1,)*(x.dim()-1), keep, device=x.device))
        return x.div(keep) * mask
```

## Conclusiones

- StochDepth regulariza redes profundas (>50 layers) y acelera entrenamiento (~25%).
- Linear schedule por capa es estándar; uniform funciona peor.
- En inference se escala por (1-p) — equivalente a `inverted dropout`.
- LayerDrop habilita inference más rápido cortando layers post-train.